In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.getOrCreate()
spark.stop()

spark = SparkSession.builder \
    .appName("GFN-EDA") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.legacy.parquetNanosAsLong", "true") \
    .getOrCreate()

print(f"Master: {spark.sparkContext.master}")
print(f"Spark UI URL: {spark.sparkContext.uiWebUrl}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/21 19:55:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Master: local[*]
Spark UI URL: http://9898a2975188:4040


In [4]:
users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")
sessions = spark.read.parquet("/home/spark/work/data/raw/session_logs.parquet")
games = spark.read.parquet("/home/spark/work/data/raw/game_catalog.parquet")
sub_events = spark.read.parquet("/home/spark/work/data/raw/subscription_events.parquet")
payments = spark.read.parquet("/home/spark/work/data/raw/payments.parquet")

User's basic EDA

In [5]:
for name, df in [("users", users), ("sessions", sessions), ("games", games),
                  ("sub_events", sub_events), ("payments", payments)]:
    print(f"=== {name}: {df.count():,} rows, {len(df.columns)} cols ===")
    df.printSchema()

=== users: 50,000 rows, 9 cols ===
root
 |-- user_id: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- subscription_tier: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- region: string (nullable = true)
 |-- referral_source: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- persona: string (nullable = true)

=== sessions: 3,553,650 rows, 15 cols ===
root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- game_id: string (nullable = true)
 |-- start_time: timestamp_ntz (nullable = true)
 |-- end_time: timestamp_ntz (nullable = true)
 |-- stream_resolution: string (nullable = true)
 |-- avg_latency_ms: double (nullable = true)
 |-- avg_fps: double (nullable = true)
 |-- total_frame_drops: long (nullable = true)
 |-- disconnect_count: long (nullable = true)
 |-- input_lag_ms: double (nullable = true)
 |-- avg_bitrate_mbps: double (nullable = true)
 |

In [5]:
users.groupBy("persona").count().orderBy("count", ascending=False).show()

+--------------+-----+
|       persona|count|
+--------------+-----+
|       regular|17409|
|        casual|15181|
|      hardcore| 9981|
|about_to_churn| 7429|
+--------------+-----+



In [6]:
users.groupBy("subscription_tier").count().orderBy("count", ascending=False).show()

+-----------------+-----+
|subscription_tier|count|
+-----------------+-----+
|         priority|19549|
|             free|18321|
|         ultimate|12130|
+-----------------+-----+



In [7]:
users.groupBy("persona") \
    .pivot("subscription_tier") \
    .count() \
    .show()

+--------------+-----+--------+--------+
|       persona| free|priority|ultimate|
+--------------+-----+--------+--------+
|        casual|10589|    3876|     716|
|      hardcore|  502|    3563|    5916|
|       regular| 3512|    9526|    4371|
|about_to_churn| 3718|    2584|    1127|
+--------------+-----+--------+--------+



In [8]:
for col in ["device_type", "region", "referral_source", "age_group", "gender"]:
    print(f"\n--- {col} ---")
    users.groupBy(col).count().orderBy("count", ascending=False).show(truncate=False)


--- device_type ---
+-----------+-----+
|device_type|count|
+-----------+-----+
|PC         |22590|
|Mobile     |9983 |
|Mac        |7516 |
|Chromebook |4971 |
|SHIELD     |4940 |
+-----------+-----+


--- region ---
+--------------+-----+
|region        |count|
+--------------+-----+
|New_Taipei    |8510 |
|Kaohsiung     |5979 |
|Taichung      |5953 |
|Taipei        |5444 |
|Taoyuan       |4967 |
|Tainan        |3898 |
|Changhua      |2540 |
|Pingtung      |1623 |
|Hsinchu_City  |1561 |
|Yunlin        |1521 |
|Hsinchu_County|1050 |
|Miaoli        |1035 |
|Nantou        |1008 |
|Yilan         |996  |
|Keelung       |982  |
|Chiayi_County |955  |
|Hualien       |522  |
|Chiayi_City   |488  |
|Taitung       |471  |
|Penghu        |252  |
+--------------+-----+
only showing top 20 rows


--- referral_source ---
+---------------+-----+
|referral_source|count|
+---------------+-----+
|organic        |19991|
|ad             |12594|
|friend_referral|10038|
|bundled        |7377 |
+----------

In [9]:
users.withColumn("signup_month", F.month(F.col("signup_date"))) \
    .groupBy("signup_month").count().orderBy("signup_month").show()

+------------+-----+
|signup_month|count|
+------------+-----+
|           1| 5202|
|           2| 4297|
|           3| 3201|
|           4| 3124|
|           5| 3592|
|           6| 3902|
|           7| 5675|
|           8| 5697|
|           9| 3860|
|          10| 3127|
|          11| 3468|
|          12| 4855|
+------------+-----+



session basic EDA

In [7]:
sessions.select(
    F.min("start_time").alias("earliest"),
    F.max("start_time").alias("latest"),
    F.count("session_id").alias("total_sessions"),
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("game_id").alias("unique_games"),
).show(truncate=False)

+-------------------+-------------------+--------------+------------+------------+
|earliest           |latest             |total_sessions|unique_users|unique_games|
+-------------------+-------------------+--------------+------------+------------+
|2024-01-01 00:00:00|2024-03-24 23:59:00|3553650       |50000       |200         |
+-------------------+-------------------+--------------+------------+------------+



In [8]:
sess_with_dur = sessions.withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

sess_with_dur.select(
    F.mean("duration_min").alias("mean"),
    F.stddev("duration_min").alias("std"),
    F.expr("percentile_approx(duration_min, 0.5)").alias("p50"),
    F.expr("percentile_approx(duration_min, 0.95)").alias("p95"),
    F.expr("percentile_approx(duration_min, 0.99)").alias("p99"),
    F.max("duration_min").alias("max"),
).show()

+------------------+-----------------+-----------------+------------------+------------------+-----------------+
|              mean|              std|              p50|               p95|               p99|              max|
+------------------+-----------------+-----------------+------------------+------------------+-----------------+
|110.07204852756855|74.84534356465736|98.33333333333333|207.16666666666666|451.96666666666664|719.9833333333333|
+------------------+-----------------+-----------------+------------------+------------------+-----------------+



In [9]:
# Join persona info
sess_persona = sess_with_dur.join(
    users.select("user_id", "persona"), on="user_id"
)

sess_persona.groupBy("persona").agg(
    F.count("session_id").alias("total_sessions"),
    F.mean("duration_min").alias("avg_duration"),
    F.mean("avg_latency_ms").alias("avg_latency"),
    F.mean("avg_fps").alias("avg_fps"),
    F.mean("disconnect_count").alias("avg_disconnects"),
).show()

+--------------+--------------+------------------+------------------+------------------+-------------------+
|       persona|total_sessions|      avg_duration|       avg_latency|           avg_fps|    avg_disconnects|
+--------------+--------------+------------------+------------------+------------------+-------------------+
|        casual|        320095| 46.62416636727642|  50.3866145987913|47.338425467439386| 0.3008825504928224|
|      hardcore|       1695907|145.91826705905996|31.509288893789183| 55.25904799024881| 0.2998366066063764|
|       regular|       1207923| 87.68456164010475|38.386077754955565|  52.7380597107602|0.30026334460060783|
|about_to_churn|        329725| 69.31037511057204|45.497626810220765| 49.55514474183063|0.49813632572598376|
+--------------+--------------+------------------+------------------+------------------+-------------------+



In [12]:
import datetime

OBS_START = datetime.date(2024, 1, 1)

sess_weekly = sess_persona.withColumn(
    "week_num",
    F.floor(F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7) + 1
)

# Average sessions per user per week, grouped by persona
decay_check = sess_weekly.groupBy("persona", "week_num") \
    .agg(F.countDistinct("user_id").alias("active_users"),
         F.count("session_id").alias("sessions")) \
    .withColumn("sessions_per_user", F.col("sessions") / F.col("active_users")) \
    .orderBy("persona", "week_num")

decay_check.filter(F.col("persona") == "about_to_churn").show(12)
decay_check.filter(F.col("persona") == "hardcore").show(12)

+--------------+--------+------------+--------+------------------+
|       persona|week_num|active_users|sessions| sessions_per_user|
+--------------+--------+------------+--------+------------------+
|about_to_churn|       1|        7162|   31701| 4.426277576096062|
|about_to_churn|       2|        7148|   31132| 4.355344152210408|
|about_to_churn|       3|        7156|   31232|4.3644494130799325|
|about_to_churn|       4|        7168|   31145| 4.345005580357143|
|about_to_churn|       5|        7137|   31292| 4.384475269721172|
|about_to_churn|       6|        7136|   31149| 4.365050448430493|
|about_to_churn|       7|        7145|   31107|  4.35367389783065|
|about_to_churn|       8|        7358|   56138|  7.62951889100299|
|about_to_churn|       9|        7299|   35754| 4.898479243732018|
|about_to_churn|      10|        6034|   12430| 2.059993370898243|
|about_to_churn|      11|        3865|    5563| 1.439327296248383|
|about_to_churn|      12|         968|    1082|1.1177685950413

+--------+--------+------------+--------+------------------+
| persona|week_num|active_users|sessions| sessions_per_user|
+--------+--------+------------+--------+------------------+
|hardcore|       1|        9981|  126958| 12.71996793908426|
|hardcore|       2|        9981|  126443| 12.66836990281535|
|hardcore|       3|        9980|  126368|12.662124248496994|
|hardcore|       4|        9981|  126371| 12.66115619677387|
|hardcore|       5|        9981|  126379|12.661957719667368|
|hardcore|       6|        9981|  126283|12.652339444945396|
|hardcore|       7|        9981|  126133|12.637310890692316|
|hardcore|       8|        9981|  221528|22.194970443843303|
|hardcore|       9|        9981|  210602|21.100290552048893|
|hardcore|      10|        9981|  126340|12.658050295561567|
|hardcore|      11|        9980|  126275|12.652805611222444|
|hardcore|      12|        9981|  126227|12.646728784690913|
+--------+--------+------------+--------+------------------+



In [10]:
sessions.groupBy("exit_type").count().orderBy("count", ascending=False).show()

sess_persona.groupBy("persona").pivot("exit_type") \
    .agg(F.count("session_id")).show()

+----------+-------+
| exit_type|  count|
+----------+-------+
|    normal|2679751|
|disconnect| 542732|
|   timeout| 168178|
|     crash| 162989|
+----------+-------+



+--------------+-----+----------+-------+-------+
|       persona|crash|disconnect| normal|timeout|
+--------------+-----+----------+-------+-------+
|        casual|14287|     47143| 243495|  15170|
|      hardcore|76110|    250043|1290585|  79169|
|       regular|54098|    178108| 919295|  56422|
|about_to_churn|18494|     67438| 226376|  17417|
+--------------+-----+----------+-------+-------+



In [13]:
sess_weekly = sess_persona.withColumn(
    "week_num",
    F.floor(F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7) + 1
)

sess_weekly.filter(F.col("week_num") >= 8) \
    .groupBy("persona").pivot("exit_type").agg(F.count("session_id")).show()

+--------------+-----+----------+------+-------+
|       persona|crash|disconnect|normal|timeout|
+--------------+-----+----------+------+-------+
|        casual| 6835|     22895|117571|   7353|
|      hardcore|36336|    119744|617201|  37691|
|       regular|25986|     85682|442472|  27563|
|about_to_churn| 8685|     35326| 59840|   7116|
+--------------+-----+----------+------+-------+



In [14]:
sess_tier = sess_with_dur.join(users.select("user_id", "subscription_tier"), on="user_id")

sess_tier.filter(F.col("subscription_tier") == "free") \
    .select(F.max("duration_min").alias("max_free_dur")).show()

+-----------------+
|     max_free_dur|
+-----------------+
|719.9833333333333|
+-----------------+



Game Catalog EDA

In [15]:
games.groupBy("popularity_tier").count().orderBy("popularity_tier").show()
games.groupBy("genre").count().orderBy("count", ascending=False).show()

+---------------+-----+
|popularity_tier|count|
+---------------+-----+
|              A|   38|
|              B|   63|
|              C|   92|
|              S|    7|
+---------------+-----+

+-------------+-----+
|        genre|count|
+-------------+-----+
|   Simulation|   33|
|     Strategy|   31|
|       Casual|   30|
|          RPG|   24|
|       Racing|   22|
|          FPS|   22|
|       Sports|   20|
|Battle_Royale|   18|
+-------------+-----+



In [16]:
sessions.join(games.select("game_id", "popularity_tier"), on="game_id") \
    .groupBy("popularity_tier").count().orderBy("popularity_tier").show()

+---------------+-------+
|popularity_tier|  count|
+---------------+-------+
|              A|1412106|
|              B| 937153|
|              C| 683828|
|              S| 520563|
+---------------+-------+



Subscription Events EDA

In [17]:
sub_events.groupBy("event_type").count().orderBy("count", ascending=False).show()

# Per persona
sub_events.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").pivot("event_type").agg(F.count("event_id")).show()

+----------+-----+
|event_type|count|
+----------+-----+
|     renew|25630|
|    cancel| 5377|
|   upgrade| 3768|
| downgrade| 3763|
+----------+-----+

+--------------+------+---------+-----+-------+
|       persona|cancel|downgrade|renew|upgrade|
+--------------+------+---------+-----+-------+
|        casual|  1487|      815| 6129|   1416|
|      hardcore|    87|      175| 7928|    191|
|       regular|   878|     1475|10451|   1982|
|about_to_churn|  2925|     1298| 1122|    179|
+--------------+------+---------+-----+-------+



In [18]:
sub_events.withColumn("event_day", F.datediff(F.col("event_date"), F.lit(OBS_START))) \
    .groupBy("event_type").agg(
        F.mean("event_day").alias("avg_day"),
        F.min("event_day").alias("min_day"),
        F.max("event_day").alias("max_day"),
    ).show()

+----------+------------------+-------+-------+
|event_type|           avg_day|min_day|max_day|
+----------+------------------+-------+-------+
|    cancel| 69.50213873907383|     56|     83|
|   upgrade|41.810509554140125|      0|     83|
|     renew| 41.47015216543114|      0|     83|
| downgrade| 62.46957214988041|     42|     83|
+----------+------------------+-------+-------+



Payments EDA

In [19]:
payments.groupBy("payment_type").agg(
    F.count("payment_id").alias("count"),
    F.mean("amount_usd").alias("avg_amount"),
    F.sum("amount_usd").alias("total_amount"),
).show()

payments.groupBy("status").count().show()

# Failed/refund rate by persona
payments.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").pivot("status").agg(F.count("payment_id")).show()

+------------+-----+------------------+------------------+
|payment_type|count|        avg_amount|      total_amount|
+------------+-----+------------------+------------------+
|    day_pass|61375|3.9899999999968956|244886.24999980946|
|      in_app|82636|2.7468614163292364|226989.63999978278|
|subscription|95037|13.819035007410017|1313319.6299992257|
+------------+-----+------------------+------------------+

+--------+------+
|  status| count|
+--------+------+
|refunded|  5674|
| success|224579|
|  failed|  8795|
+--------+------+

+--------------+------+--------+-------+
|       persona|failed|refunded|success|
+--------------+------+--------+-------+
|        casual|  3023|    1774|  54744|
|      hardcore|  1394|     738|  67013|
|       regular|  2778|    1783|  89209|
|about_to_churn|  1600|    1379|  13613|
+--------------+------+--------+-------+



#### Features Engineering

##### Data loading

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import datetime

spark = SparkSession.builder \
    .appName("gfn-feature-engineering") \
    .master("local[*]") \
    .getOrCreate()

users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")
sessions = spark.read.parquet("/home/spark/work/data/raw/session_logs.parquet")
games = spark.read.parquet("/home/spark/work/data/raw/game_catalog.parquet")
sub_events = spark.read.parquet("/home/spark/work/data/raw/subscription_events.parquet")
payments = spark.read.parquet("/home/spark/work/data/raw/payments.parquet")

##### Temporal constants

In [31]:
# Temporal windows
DATA_START    = datetime.date(2024, 1, 1)    # Week 1 start
BASELINE_END  = datetime.date(2024, 1, 29)   # Week 5 start = baseline end
OBS_START     = datetime.date(2024, 1, 29)   # Observation window start (week 5)
OBS_END       = datetime.date(2024, 2, 26)   # Observation window end (week 9 start, exclusive)
BASELINE_WEEKS = 4

In [32]:
# ── Shared base: obs_sessions with week_num and duration ──
obs_sessions = sessions.filter(
    (F.col("start_time").cast("date") >= F.lit(OBS_START)) &
    (F.col("start_time").cast("date") < F.lit(OBS_END))
).withColumn(
    "week_num",
    (F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7).cast("int") + 1
).withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

##### Session Patterns

In [33]:
# ══════════════════════════════════════════════
# Section 1: Session Patterns
# Output: session_features (user_id, week_num, 6 features)
# ══════════════════════════════════════════════

# Core aggregations
session_patterns = obs_sessions.groupBy("user_id", "week_num").agg(
    F.count("session_id").alias("weekly_session_count"),
    F.avg("duration_min").alias("avg_session_duration_min"),
    F.sum("duration_min").alias("total_playtime_min"),
    F.avg(
        F.when(F.hour("start_time").between(19, 23), 1).otherwise(0)
    ).alias("peak_hour_ratio"),
    F.avg(
        F.when(F.dayofweek("start_time").isin(1, 7), 1).otherwise(0)
    ).alias("weekend_ratio"),
)

# Session regularity: std of inter-session gaps (computed separately due to lag)
w_session_order = Window.partitionBy("user_id", "week_num").orderBy("start_time")

session_regularity = obs_sessions.withColumn(
    "prev_end", F.lag("end_time").over(w_session_order)
).withColumn(
    "inter_session_gap_min",
    (F.unix_timestamp("start_time") - F.unix_timestamp("prev_end")) / 60.0
).filter(
    F.col("inter_session_gap_min").isNotNull()
).groupBy("user_id", "week_num").agg(
    F.stddev("inter_session_gap_min").alias("session_regularity")
)

# Combine into section output
session_features = session_patterns.join(
    session_regularity, on=["user_id", "week_num"], how="left"
).fillna(0, subset=["session_regularity"])

In [34]:
# ── Section 1 validation ──
print(f"session_features: {session_features.count():,} rows")
session_features.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").agg(
        F.avg("weekly_session_count").alias("avg_sessions"),
        F.avg("avg_session_duration_min").alias("avg_duration"),
        F.avg("session_regularity").alias("avg_regularity"),
        F.avg("peak_hour_ratio").alias("avg_peak"),
        F.avg("weekend_ratio").alias("avg_weekend"),
    ).show()

session_features: 186,144 rows


+--------------+------------------+------------------+------------------+-------------------+-------------------+
|       persona|      avg_sessions|      avg_duration|    avg_regularity|           avg_peak|        avg_weekend|
+--------------+------------------+------------------+------------------+-------------------+-------------------+
|        casual|2.2980041995392737| 45.94846922018573| 656.8402621487083| 0.6946300066109236| 0.6017398414866449|
|      hardcore|14.997144574691914|146.05991828459003|  722.913002609727|0.49504290919636695|0.39998097168006785|
|       regular|6.2729803802149045| 87.54242533792937|1303.4443145074695| 0.5946212917070854|  0.498888568980365|
|about_to_churn| 3.781039495142877| 60.90499137152312|1209.1851991539784| 0.5944587358169573| 0.4970872356428622|
+--------------+------------------+------------------+------------------+-------------------+-------------------+



##### Engagement Decay

In [48]:
# ══════════════════════════════════════════════
# Section 2: Engagement Decay
# Input dependency: session_features (for wow_change and baseline ratios)
# Output: engagement_features (user_id, week_num, 6 features)
# ══════════════════════════════════════════════

# --- Scaffold: ensure every user has all 4 weeks ---
all_user_weeks = obs_sessions.select("user_id").distinct().crossJoin(
    spark.createDataFrame([(i,) for i in range(1, 5)], ["week_num"])
)

session_features_filled = all_user_weeks.join(
    session_features, on=["user_id", "week_num"], how="left"
).fillna(0, subset=[
    "weekly_session_count", "total_playtime_min",
    "avg_session_duration_min", "peak_hour_ratio",
    "weekend_ratio", "session_regularity",
])

# --- 2a. Week-over-week change (now on filled data) ---
wow_window = Window.partitionBy("user_id").orderBy("week_num")

wow_base = session_features_filled.select(
    "user_id", "week_num", "weekly_session_count", "total_playtime_min"
).withColumn(
    "prev_session_count", F.lag("weekly_session_count").over(wow_window)
).withColumn(
    "prev_playtime", F.lag("total_playtime_min").over(wow_window)
)

wow_features = wow_base.select(
    "user_id", "week_num",
    F.when(
        F.col("prev_session_count").isNull() | (F.col("prev_session_count") == 0),
        F.lit(None)
    ).otherwise(
        (F.col("weekly_session_count") - F.col("prev_session_count"))
        / F.col("prev_session_count")
    ).alias("session_count_wow_change"),

    F.when(
        F.col("prev_playtime").isNull() | (F.col("prev_playtime") == 0),
        F.lit(None)
    ).otherwise(
        (F.col("total_playtime_min") - F.col("prev_playtime"))
        / F.col("prev_playtime")
    ).alias("playtime_wow_change"),
)

# --- 2b. Baseline comparison (also on filled data) ---
baseline_sessions = sessions.filter(
    (F.col("start_time").cast("date") >= F.lit(DATA_START)) &
    (F.col("start_time").cast("date") < F.lit(BASELINE_END))
).withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

user_baseline = baseline_sessions.groupBy("user_id").agg(
    (F.count("session_id") / F.lit(BASELINE_WEEKS)).alias("baseline_avg_session_count"),
    (F.sum("duration_min") / F.lit(BASELINE_WEEKS)).alias("baseline_avg_playtime"),
)

baseline_ratios = session_features_filled.select(
    "user_id", "week_num", "weekly_session_count", "total_playtime_min"
).join(user_baseline, on="user_id", how="left")

baseline_features = baseline_ratios.select(
    "user_id", "week_num",
    F.when(
        F.col("baseline_avg_session_count").isNull() | (F.col("baseline_avg_session_count") == 0),
        F.lit(None)
    ).otherwise(
        F.col("weekly_session_count") / F.col("baseline_avg_session_count")
    ).alias("session_count_vs_baseline"),

    F.when(
        F.col("baseline_avg_playtime").isNull() | (F.col("baseline_avg_playtime") == 0),
        F.lit(None)
    ).otherwise(
        F.col("total_playtime_min") / F.col("baseline_avg_playtime")
    ).alias("playtime_vs_baseline"),
)

# --- 2c. Longest inactive days (unchanged) ---
obs_dates = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{OBS_START}'),
        date_sub(to_date('{OBS_END}'), 1),
        interval 1 day
    )) AS date
""").withColumn(
    "week_num",
    (F.datediff(F.col("date"), F.lit(OBS_START)) / 7).cast("int") + 1
)

obs_user_ids = obs_sessions.select("user_id").distinct()
user_date_scaffold = obs_user_ids.crossJoin(obs_dates)

session_dates = obs_sessions.select(
    "user_id",
    F.col("start_time").cast("date").alias("date")
).distinct().withColumn("had_session", F.lit(1))

daily_activity = user_date_scaffold.join(
    session_dates, on=["user_id", "date"], how="left"
).fillna(0, subset=["had_session"])

streak_window = Window.partitionBy("user_id").orderBy("date")
daily_activity = daily_activity.withColumn(
    "session_cumsum", F.sum("had_session").over(streak_window)
)

inactive_streaks = daily_activity.filter(F.col("had_session") == 0)
streak_group_window = Window.partitionBy("user_id", "session_cumsum").orderBy("date")
inactive_streaks = inactive_streaks.withColumn(
    "streak_len", F.row_number().over(streak_group_window)
)

longest_inactive = inactive_streaks.groupBy("user_id", "week_num").agg(
    F.max("streak_len").alias("longest_inactive_days")
)

# --- 2d. Combine all Section 2 features ---
engagement_features = wow_features \
    .join(baseline_features, on=["user_id", "week_num"], how="outer") \
    .join(longest_inactive,  on=["user_id", "week_num"], how="outer") \
    .fillna(0, subset=["longest_inactive_days"])

##### Streaming Quality

In [40]:
# ══════════════════════════════════════════════
# Section 3: Streaming Quality
# Output: streaming_features (user_id, week_num, 8 features)
# ══════════════════════════════════════════════

streaming_features = obs_sessions.groupBy("user_id", "week_num").agg(
    F.avg("avg_latency_ms").alias("avg_latency"),
    F.avg("avg_fps").alias("avg_fps"),
    F.avg("total_frame_drops").alias("frame_drop_rate"),
    F.avg("disconnect_count").alias("disconnect_rate"),
    F.avg("avg_bitrate_mbps").alias("avg_bitrate"),
    F.avg("avg_jitter_ms").alias("avg_jitter"),
    F.avg("packet_loss_rate").alias("packet_loss_avg"),
    F.avg(
        F.when(F.col("exit_type").isin("crash", "disconnect", "timeout"), 1).otherwise(0)
    ).alias("crash_exit_ratio"),
)

In [41]:
# ── Section 3 validation ──
print(f"streaming_features: {streaming_features.count():,} rows")

# Quality should correlate with tier (join via users for persona proxy)
streaming_features.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").agg(
        F.avg("avg_latency").alias("avg_latency"),
        F.avg("avg_fps").alias("avg_fps"),
        F.avg("crash_exit_ratio").alias("crash_exit_ratio"),
    ).show()

# about_to_churn week-level decay check
streaming_features.join(users.select("user_id", "persona"), on="user_id") \
    .filter(F.col("persona") == "about_to_churn") \
    .groupBy("week_num").agg(
        F.avg("avg_latency").alias("avg_latency"),
        F.avg("crash_exit_ratio").alias("crash_exit_ratio"),
    ).orderBy("week_num").show()

streaming_features: 186,144 rows


+--------------+------------------+------------------+-------------------+
|       persona|       avg_latency|           avg_fps|   crash_exit_ratio|
+--------------+------------------+------------------+-------------------+
|        casual|50.424443653195574| 47.32795640815108|0.25098920156714694|
|      hardcore| 31.50185394781748| 55.25293552679558| 0.2429556223036173|
|       regular|38.398406619334914|52.771167834433264| 0.2492668532943302|
|about_to_churn| 45.57443766045496| 49.55479585681164| 0.4758125183726119|
+--------------+------------------+------------------+-------------------+

+--------+-----------------+-------------------+
|week_num|      avg_latency|   crash_exit_ratio|
+--------+-----------------+-------------------+
|       1|45.47183139571818| 0.2591148481924157|
|       2|45.21685119110733|0.25066596182761275|
|       3|45.30639714537124| 0.7035673146148302|
|       4|46.31436648173614| 0.7024160116567811|
+--------+-----------------+-------------------+



##### Game Diversity

In [ ]:
obs_sessions.show(5, truncate=False)

+----------+-------+-------+-------------------+--------------------------+-----------------+--------------+-------+-----------------+----------------+------------+----------------+-------------+----------------+----------+--------+------------------+
|session_id|user_id|game_id|start_time         |end_time                  |stream_resolution|avg_latency_ms|avg_fps|total_frame_drops|disconnect_count|input_lag_ms|avg_bitrate_mbps|avg_jitter_ms|packet_loss_rate|exit_type |week_num|duration_min      |
+----------+-------+-------+-------------------+--------------------------+-----------------+--------------+-------+-----------------+----------------+------------+----------------+-------------+----------------+----------+--------+------------------+
|S00958932 |U013659|G0116  |2024-01-29 00:00:00|2024-01-29 00:22:35.429957|720p             |74.2          |61.4   |2946             |1               |92.8        |13.0            |57.2         |0.0823          |disconnect|1       |22.583333333

In [67]:
# ══════════════════════════════════════════════
# Section 4: Game Diversity
# Output: game_features (user_id, week_num, 4 features)
# ══════════════════════════════════════════════

# --- 4a. unique_games_played & top_game_concentration ---
game_playtime = obs_sessions.groupBy("user_id", "week_num", "game_id").agg(
    F.sum("duration_min").alias("game_playtime"),
    F.count("session_id").alias("game_sessions"),
)

# Top game concentration: % of playtime on most-played game
w_game = Window.partitionBy("user_id", "week_num")

game_concentration = game_playtime.withColumn(
    "total_playtime", F.sum("game_playtime").over(w_game)
).withColumn(
    "playtime_share", F.col("game_playtime") / F.col("total_playtime")
)

game_basic = game_concentration.groupBy("user_id", "week_num").agg(
    F.countDistinct("game_id").alias("unique_games_played"),
    F.max("playtime_share").alias("top_game_concentration"),
)

In [68]:
# --- 4b. genre_entropy ---
# Shannon entropy of genre distribution per user per week
# H = -Σ p(g) * log2(p(g))

game_with_genre = obs_sessions.join(
    games.select("game_id", "genre"), on="game_id", how="left"
)

genre_counts = game_with_genre.groupBy("user_id", "week_num", "genre").agg(
    F.count("session_id").alias("genre_sessions")
)

w_genre = Window.partitionBy("user_id", "week_num")

genre_probs = genre_counts.withColumn(
    "total_sessions", F.sum("genre_sessions").over(w_genre)
).withColumn(
    "p", F.col("genre_sessions") / F.col("total_sessions")
).withColumn(
    "neg_p_log2_p", -F.col("p") * F.log2(F.col("p"))
)

genre_entropy = genre_probs.groupBy("user_id", "week_num").agg(
    F.sum("neg_p_log2_p").alias("genre_entropy")
)

In [69]:
# --- 4c. new_game_trial_rate ---
# % of sessions on games not played in any prior week within obs window
# week 1 = null (no prior history in obs window to compare)

# All games each user has played up to (but not including) each week
w_cumulative = Window.partitionBy("user_id").orderBy("week_num") \
    .rowsBetween(Window.unboundedPreceding, -1)

# Step 1: Collect distinct games per user per week
user_week_games = obs_sessions.groupBy("user_id", "week_num").agg(
    F.collect_set("game_id").alias("current_week_games")
)

# Step 2: Collect all games played in prior weeks
# Use a self-join approach: for each (user, week), find all games from earlier weeks
prior_games = obs_sessions.select("user_id", "week_num", "game_id").distinct()

prior_games_agg = prior_games.alias("a").join(
    prior_games.alias("b"),
    (F.col("a.user_id") == F.col("b.user_id")) &
    (F.col("a.week_num") > F.col("b.week_num"))
).groupBy(
    F.col("a.user_id").alias("user_id"),
    F.col("a.week_num").alias("week_num")
).agg(
    F.collect_set(F.col("b.game_id")).alias("prior_games")
)

# Step 3: Compute new game trial rate
new_game_rate = user_week_games.join(
    prior_games_agg, on=["user_id", "week_num"], how="left"
).withColumn(
    "new_games",
    F.when(
        F.col("prior_games").isNull(),
        F.lit(None)  # week 1: no prior data → null
    ).otherwise(
        F.size(F.array_except(F.col("current_week_games"), F.col("prior_games")))
    )
).withColumn(
    "new_game_trial_rate",
    F.when(
        F.col("new_games").isNull(),
        F.lit(None)
    ).otherwise(
        F.col("new_games") / F.size(F.col("current_week_games"))
    )
).select("user_id", "week_num", "new_game_trial_rate")

In [70]:
# --- 4d. Combine all Section 4 features ---
game_features = game_basic \
    .join(genre_entropy,  on=["user_id", "week_num"], how="outer") \
    .join(new_game_rate,  on=["user_id", "week_num"], how="outer")

In [71]:
# ── Section 4 validation ──
print(f"game_features: {game_features.count():,} rows")
print(f"Columns: {game_features.columns}")

game_features.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").agg(
        F.avg("unique_games_played").alias("avg_unique_games"),
        F.avg("genre_entropy").alias("avg_entropy"),
        F.avg("new_game_trial_rate").alias("avg_new_trial"),
        F.avg("top_game_concentration").alias("avg_concentration"),
    ).show()

# Expected:
# - hardcore: highest unique_games, highest entropy (plays everything)
# - casual: low unique_games, low entropy (sticks to 1-2 genres)
# - about_to_churn: moderate but declining over weeks
# - top_game_concentration: casual highest (few games), hardcore lowest

game_features: 186,144 rows
Columns: ['user_id', 'week_num', 'unique_games_played', 'top_game_concentration', 'genre_entropy', 'new_game_trial_rate']


26/03/22 04:36:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 04:36:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 04:36:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 04:36:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------+------------------+------------------+------------------+-------------------+
|       persona|  avg_unique_games|       avg_entropy|     avg_new_trial|  avg_concentration|
+--------------+------------------+------------------+------------------+-------------------+
|        casual|2.2784335310786292|0.8139459363221836|0.9671608346030368| 0.6770259802795207|
|      hardcore|14.016907123534716|2.4856481184146846|0.8214899983768611|0.15651013726453894|
|       regular| 6.092704572149476|1.8487829728316767|0.9163744749462168|0.32447059500067543|
|about_to_churn|3.7225058498191874|1.3596525436591476| 0.932131802225552| 0.4739607132759307|
+--------------+------------------+------------------+------------------+-------------------+



##### Final Join

In [72]:
# ══════════════════════════════════════════════
# Final: Merge all feature sections
# ══════════════════════════════════════════════

weekly_features = session_features_filled \
    .join(engagement_features, on=["user_id", "week_num"], how="outer") \
    .join(streaming_features,  on=["user_id", "week_num"], how="outer") \
    .join(game_features,       on=["user_id", "week_num"], how="outer")  # Section 4
    # .join(volatility_features, on=["user_id", "week_num"], how="outer")  # Section 5
    # .join(payment_features,    on=["user_id", "week_num"], how="outer")  # Section 6

print(f"weekly_features: {weekly_features.count():,} rows, {len(weekly_features.columns)} cols")
print(f"Columns: {weekly_features.columns}")

weekly_features: 199,384 rows, 25 cols
Columns: ['user_id', 'week_num', 'weekly_session_count', 'avg_session_duration_min', 'total_playtime_min', 'peak_hour_ratio', 'weekend_ratio', 'session_regularity', 'session_count_wow_change', 'playtime_wow_change', 'session_count_vs_baseline', 'playtime_vs_baseline', 'longest_inactive_days', 'avg_latency', 'avg_fps', 'frame_drop_rate', 'disconnect_rate', 'avg_bitrate', 'avg_jitter', 'packet_loss_avg', 'crash_exit_ratio', 'unique_games_played', 'top_game_concentration', 'genre_entropy', 'new_game_trial_rate']


In [73]:
weekly_features.show(5, truncate=False)

26/03/22 20:08:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 20:08:20 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+--------+--------------------+------------------------+------------------+---------------+-------------+------------------+------------------------+--------------------+-------------------------+--------------------+---------------------+-----------+-----------------+---------------+------------------+------------------+-----------------+--------------------+----------------+-------------------+----------------------+-----------------+-------------------+
|user_id|week_num|weekly_session_count|avg_session_duration_min|total_playtime_min|peak_hour_ratio|weekend_ratio|session_regularity|session_count_wow_change|playtime_wow_change |session_count_vs_baseline|playtime_vs_baseline|longest_inactive_days|avg_latency|avg_fps          |frame_drop_rate|disconnect_rate   |avg_bitrate       |avg_jitter       |packet_loss_avg     |crash_exit_ratio|unique_games_played|top_game_concentration|genre_entropy    |new_game_trial_rate|
+-------+--------+--------------------+-----------------------

In [50]:
# ══════════════════════════════════════════════
# Validation Level 1: Structural integrity
# ══════════════════════════════════════════════

total_users = users.count()
print(f"Total users: {total_users:,}")
print(f"weekly_features rows: {weekly_features.count():,}")
print(f"Unique users in weekly_features: {weekly_features.select('user_id').distinct().count():,}")
print(f"Columns ({len(weekly_features.columns)}): {weekly_features.columns}")
print()

# Key uniqueness: no duplicate (user_id, week_num) pairs
dup_check = weekly_features.groupBy("user_id", "week_num").count().filter(F.col("count") > 1)
print(f"Duplicate (user_id, week_num) pairs: {dup_check.count()}")
print()

# Week distribution: should be 4 weeks, roughly equal
weekly_features.groupBy("week_num").count().orderBy("week_num").show()

# Null counts per column
print("Null counts:")
weekly_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in weekly_features.columns
]).show(vertical=True)

Total users: 50,000
weekly_features rows: 199,384
Unique users in weekly_features: 49,846
Columns (21): ['user_id', 'week_num', 'weekly_session_count', 'avg_session_duration_min', 'total_playtime_min', 'peak_hour_ratio', 'weekend_ratio', 'session_regularity', 'session_count_wow_change', 'playtime_wow_change', 'session_count_vs_baseline', 'playtime_vs_baseline', 'longest_inactive_days', 'avg_latency', 'avg_fps', 'frame_drop_rate', 'disconnect_rate', 'avg_bitrate', 'avg_jitter', 'packet_loss_avg', 'crash_exit_ratio']

Duplicate (user_id, week_num) pairs: 0

+--------+-----+
|week_num|count|
+--------+-----+
|       1|49846|
|       2|49846|
|       3|49846|
|       4|49846|
+--------+-----+

Null counts:


26/03/22 02:55:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 02:55:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/22 02:55:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


-RECORD 0--------------------------
 user_id                   | 0     
 week_num                  | 0     
 weekly_session_count      | 0     
 avg_session_duration_min  | 0     
 total_playtime_min        | 0     
 peak_hour_ratio           | 0     
 weekend_ratio             | 0     
 session_regularity        | 0     
 session_count_wow_change  | 61716 
 playtime_wow_change       | 61716 
 session_count_vs_baseline | 1276  
 playtime_vs_baseline      | 1276  
 longest_inactive_days     | 0     
 avg_latency               | 13240 
 avg_fps                   | 13240 
 frame_drop_rate           | 13240 
 disconnect_rate           | 13240 
 avg_bitrate               | 13240 
 avg_jitter                | 13240 
 packet_loss_avg           | 13240 
 crash_exit_ratio          | 13240 



In [51]:
# ══════════════════════════════════════════════
# Validation Level 2: Value range sanity
# ══════════════════════════════════════════════

# Section 1: Session Patterns
weekly_features.select(
    F.min("weekly_session_count").alias("min"),
    F.expr("percentile_approx(weekly_session_count, 0.5)").alias("p50"),
    F.max("weekly_session_count").alias("max"),
).show()
# Expected: min >= 1, max could be 30+ for hardcore during hot weeks

weekly_features.select(
    F.min("avg_session_duration_min").alias("min"),
    F.expr("percentile_approx(avg_session_duration_min, 0.5)").alias("p50"),
    F.expr("percentile_approx(avg_session_duration_min, 0.99)").alias("p99"),
    F.max("avg_session_duration_min").alias("max"),
).show()
# Expected: min ~5 (quality_impact floor), max could be 300+ (AFK noise)

# peak_hour_ratio and weekend_ratio should be between 0 and 1
weekly_features.select(
    F.min("peak_hour_ratio").alias("peak_min"),
    F.max("peak_hour_ratio").alias("peak_max"),
    F.min("weekend_ratio").alias("wknd_min"),
    F.max("weekend_ratio").alias("wknd_max"),
).show()
# Expected: both [0.0, 1.0]

# Section 2: wow_change can be negative, vs_baseline can be > 1
weekly_features.filter(F.col("week_num") > 1).select(
    F.expr("percentile_approx(session_count_wow_change, 0.01)").alias("wow_p01"),
    F.expr("percentile_approx(session_count_wow_change, 0.5)").alias("wow_p50"),
    F.expr("percentile_approx(session_count_wow_change, 0.99)").alias("wow_p99"),
).show()
# Expected: p50 near 0 (most users stable), tails can be -1.0 to +2.0+

# Section 3: Streaming quality ranges
weekly_features.select(
    F.min("avg_latency").alias("lat_min"),
    F.expr("percentile_approx(avg_latency, 0.5)").alias("lat_p50"),
    F.max("avg_latency").alias("lat_max"),    # noise spikes up to 400ms
    F.min("avg_fps").alias("fps_min"),
    F.expr("percentile_approx(avg_fps, 0.5)").alias("fps_p50"),
    F.max("avg_fps").alias("fps_max"),
).show()
# Expected: latency p50 ~30-50ms (mix of tiers), max could be 200+ (spike noise)
#           fps p50 ~50, min could be very low (AFK noise: 1-10 fps)

# crash_exit_ratio should be [0, 1]
weekly_features.select(
    F.min("crash_exit_ratio").alias("min"),
    F.expr("percentile_approx(crash_exit_ratio, 0.5)").alias("p50"),
    F.max("crash_exit_ratio").alias("max"),
).show()
# Expected: p50 ~0.15 (normal exit weights: 15% non-normal), max = 1.0

+---+---+---+
|min|p50|max|
+---+---+---+
|  0|  5| 45|
+---+---+---+



+---+-----------------+-----------------+------+
|min|              p50|              p99|   max|
+---+-----------------+-----------------+------+
|0.0|68.07083333333333|212.2888888888889|719.25|
+---+-----------------+-----------------+------+



+--------+--------+--------+--------+
|peak_min|peak_max|wknd_min|wknd_max|
+--------+--------+--------+--------+
|     0.0|     1.0|     0.0|     1.0|
+--------+--------+--------+--------+



+-------+-------+-------+
|wow_p01|wow_p50|wow_p99|
+-------+-------+-------+
|   -1.0|    0.0|    3.0|
+-------+-------+-------+



+-------+-----------------+-------+-------+-----------------+-------+
|lat_min|          lat_p50|lat_max|fps_min|          fps_p50|fps_max|
+-------+-----------------+-------+-------+-----------------+-------+
|    5.0|33.82222222222222|  399.5|    1.0|53.38947368421052|   87.6|
+-------+-----------------+-------+-------+-----------------+-------+



+---+----+---+
|min| p50|max|
+---+----+---+
|0.0|0.25|1.0|
+---+----+---+



In [52]:
# ══════════════════════════════════════════════
# Validation Level 3: Persona behavior profiles
# Compare against data_design.md generation parameters
# ══════════════════════════════════════════════

persona_stats = weekly_features.join(
    users.select("user_id", "persona", "subscription_tier"), on="user_id"
)

# --- Section 1: Session patterns by persona ---
persona_stats.groupBy("persona").agg(
    F.avg("weekly_session_count").alias("avg_sessions"),
    F.avg("avg_session_duration_min").alias("avg_duration"),
    F.avg("total_playtime_min").alias("avg_playtime"),
    F.avg("peak_hour_ratio").alias("avg_peak"),
    F.avg("weekend_ratio").alias("avg_weekend"),
).show()

+--------------+------------------+------------------+------------------+------------------+------------------+
|       persona|      avg_sessions|      avg_duration|      avg_playtime|          avg_peak|       avg_weekend|
+--------------+------------------+------------------+------------------+------------------+------------------+
|        casual|1.8718698106941216|37.427935248385396| 86.17065565150011|0.5658200882478537|0.4901551717775558|
|      hardcore|14.997144574691914| 146.0599182845901|2191.8579697258087|0.4950429091963674|0.3999809716800684|
|       regular|6.2139594346127325| 86.71875996507907| 544.5690542882868|0.5890266446996757|0.4941946478604771|
|about_to_churn|3.6000540102619496| 57.98967683719893|227.22096442074005|0.5660040204716817| 0.473293362427173|
+--------------+------------------+------------------+------------------+------------------+------------------+



In [53]:
# --- Section 2: Engagement decay by persona ---
persona_stats.groupBy("persona").agg(
    F.avg("session_count_wow_change").alias("avg_wow_change"),
    F.avg("session_count_vs_baseline").alias("avg_vs_baseline"),
    F.avg("playtime_vs_baseline").alias("avg_play_vs_base"),
    F.avg("longest_inactive_days").alias("avg_inactive"),
).show()

26/03/22 02:56:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------+--------------------+------------------+------------------+------------------+
|       persona|      avg_wow_change|   avg_vs_baseline|  avg_play_vs_base|      avg_inactive|
+--------------+--------------------+------------------+------------------+------------------+
|        casual|   0.234491330205493|1.4072383734040999|1.7605870146500693| 6.584988375954833|
|      hardcore| 0.26644941783228054|1.1896295695445527|1.1970691117745909|0.8144474501552951|
|       regular| 0.35259152154530704| 1.222617118041274|1.2603089265007528|2.5397178809469088|
|about_to_churn|-0.01296039438190...|0.8805455605659872|0.7900629563764984|3.8362476370510397|
+--------------+--------------------+------------------+------------------+------------------+



In [54]:
# ══════════════════════════════════════════════
# Validation Level 4: Churn decay progression
# The most critical check — this is what the model must learn
# ══════════════════════════════════════════════

# about_to_churn: week-by-week trend across ALL feature sections
churn_users = persona_stats.filter(F.col("persona") == "about_to_churn")

churn_weekly_trend = churn_users.groupBy("week_num").agg(
    # Section 1
    F.avg("weekly_session_count").alias("sessions"),
    F.avg("avg_session_duration_min").alias("duration"),

    # Section 2
    F.avg("session_count_wow_change").alias("wow_change"),
    F.avg("playtime_vs_baseline").alias("vs_baseline"),
    F.avg("longest_inactive_days").alias("inactive_days"),

    # Section 3
    F.avg("crash_exit_ratio").alias("crash_ratio"),
    F.avg("disconnect_rate").alias("disconnect_rate"),
).orderBy("week_num")

churn_weekly_trend.show()

26/03/22 02:58:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------+------------------+------------------+-------------------+------------------+-----------------+-------------------+-------------------+
|week_num|          sessions|          duration|         wow_change|       vs_baseline|    inactive_days|        crash_ratio|    disconnect_rate|
+--------+------------------+------------------+-------------------+------------------+-----------------+-------------------+-------------------+
|       1| 4.250337564137186|  70.5350022853368|               NULL|1.0694330069990337|2.951525789900081|0.25911484819241615| 0.3097021023855403|
|       2| 4.249662435862814|  70.0902058021706|0.07184434441247854|1.0646511020301892|3.461112611396165|0.25066596182761314|0.31340997700689766|
|       3|2.8223062381852553|49.086123267885185|-0.2826542289838301|0.5272154018429327|4.330002700513098| 0.7035673146148312| 1.5197195652919193|
|       4| 3.077909802862544| 42.24737599340304|0.17713097237935246|0.4989523146338344|4.602349446394815| 0.7024160116567822

In [55]:
# Contrast with hardcore (should be flat/stable across all weeks)
hardcore_users = persona_stats.filter(F.col("persona") == "hardcore")

hardcore_weekly_trend = hardcore_users.groupBy("week_num").agg(
    F.avg("weekly_session_count").alias("sessions"),
    F.avg("avg_session_duration_min").alias("duration"),
    F.avg("crash_exit_ratio").alias("crash_ratio"),
    F.avg("longest_inactive_days").alias("inactive_days"),
).orderBy("week_num")

hardcore_weekly_trend.show()

+--------+------------------+------------------+-------------------+------------------+
|week_num|          sessions|          duration|        crash_ratio|     inactive_days|
+--------+------------------+------------------+-------------------+------------------+
|       1|12.633503656948202|146.17153502844246|0.24100629517810915|0.9583208095381225|
|       2| 12.62819356777878|145.54555176284106|0.24399205898447474|0.9642320408776676|
|       3|12.621781384630799| 146.2547667762601|0.24574672328700242|0.9784590722372508|
|       4| 22.10509968940988| 146.2678195708172|0.24107741176488207|0.3567778779681395|
+--------+------------------+------------------+-------------------+------------------+

